In [ ]:
import os
import sys

print(os.getcwd())
# 1. Navigate UP two levels to the project root:
#    preparation/ -> CST-Part-II-Project-Code/
os.chdir('..') # Go up one level (out of preparation)

# 2. Check the path again. We should now be in the project root.
#    You don't need to go up another level, as the previous failed attempt
#    must have been a result of not resetting the environment first.
#    We will confirm and then add the current directory ('.').

# 3. Add the project root (the directory containing 'src') to sys.path
#    The '.' refers to the Current Working Directory, which is now the project root.
if '.' not in sys.path:
    sys.path.append('.')

# Check the final working directory (optional, but helpful for confirmation)
print(f"New CWD for imports: {os.getcwd()}")
print("System path successfully configured.")

: 

In [ ]:
%pip install stim
%pip install svgwrite

In [ ]:
from src.color_code_simulator.color_code_circuits.color_code_circuit_666 import ColorCodeCircuit666

In [ ]:
circ = ColorCodeCircuit666(3,1)
layout = circ.generate_layout()

In [ ]:
from dataclasses import dataclass
import stim
@dataclass
class ColorCodeTile:
    qubits: list
    ancilla: tuple
    color: str
class ColorCodeCircuit666:
    def within_bounds(self,x,y,side):
        if y<0:
            return False
        if x<y:
            return False
        if y>(side-1)-x:
            return False
        return True
        
    def generate_layout(self, distance:int):
        side = distance*3 - 2
        xtype = [['D', 'D', 'M'], ['M','D','D'], ['D','M','D']]
        dirs = [(-1,-1), (1,-1),(2,0),(1,1),(-1,1),(-2,0)]
        tiles = []
        validpos = set()
        for y in range(0, side):
            xpattern = xtype[y % 3]
            patternptr = 0
            for x in range(y,side-y,2):
                print(x,y)
                currtype = xpattern[patternptr % 3]
                if currtype == 'M':
                    tile = ColorCodeTile(
                        qubits = [(x+dx, y+dy) for dx,dy in dirs if self.within_bounds(x+dx,y+dy,side)],
                        ancilla = (x,y),
                        color = ['red','green','blue'][y % 3])
                    tiles.append(tile)
                patternptr += 1
        return tiles

    def build_circuit(self, rounds:int, distance:int, after_clifford_depolarization:float):
        tiles = self.generate_layout(distance)
        circ = stim.Circuit()

        qubits = {q for tile in tiles for q in tile.qubits}
        ancillae = {tile.ancilla for tile in tiles}
        sorted_q_a = sorted(qubits | ancillae)
        qa_index_map = {q:i for i,q in enumerate(sorted_q_a)}
        color_tiles = {'red':[tile for tile in tiles if tile.color=='red'],
                       'green':[tile for tile in tiles if tile.color=='green'],
                       'blue':[tile for tile in tiles if tile.color=="blue"]
                       }
        
        # append coords to the circuit
        for q,i in qa_index_map.items():
            circ.append("QUBIT_COORDS", [i], [q[0], q[1]])

        # reset all qubits + ancillae
        circ.append("R", qa_index_map.values())

        # repeated rounds of measurements
        inner_loop = stim.Circuit()
       
        
        
        for color, tilelist in color_tiles.items():
            ancilla_idxs = [qa_index_map[tile.ancilla] for tile in color_tiles[color]]
            #Reset ancilla qubits
            inner_loop.append("R", ancilla_idxs)
            inner_loop.append("TICK")
            # prepare ancilla qubits in X basis then measure with CNOTs
            inner_loop.append("H", ancilla_idxs)
            inner_loop.append("TICK")
            
            #perform X measurements for each tile
            for tile in tilelist:
                ancilla_idx = qa_index_map[tile.ancilla]
                for q in tile.qubits:
                    inner_loop.append("CNOT", [ancilla_idx, qa_index_map[q]])
                    inner_loop.append("TICK")
            
            #measure
            inner_loop.append("M", ancilla_idxs)###Look up
            
            
            #Reset ancilla qubits
            inner_loop.append("R", ancilla_idxs)
            inner_loop.append("TICK")
            # prepare ancilla qubits in Z basis then measure 
            
            #perform Z stabiliser measurements for each tile
            for tile in tilelist:
                ancilla_idx = qa_index_map[tile.ancilla]
                for q in tile.qubits:
                    inner_loop.append("CNOT", [qa_index_map[q], ancilla_idx])
                    inner_loop.append("TICK")
            #measure
            inner_loop.append("M", ancilla_idxs)
                    
            ##include detector logic + work out indexing
        
        circ += rounds * inner_loop
        
        return circ
circ = ColorCodeCircuit666()
layout = circ.generate_layout(3)
circ.build_circuit(2,3,0.001)

In [ ]:
import svgwrite
from IPython.display import SVG, display
color_map = {
    'red':'crimson',
    'green':'lime',
    'blue':'royalblue'
    }

def draw_tiles(tiles):
    width = max(x for tile in tiles for x,y in tile.qubits)
    height = max(y for tile in tiles for x,y in tile.qubits)
    drawing = svgwrite.Drawing(size = (width*20, height*20))

    for tile in tiles:
        drawing.add(drawing.polygon(
            points = [(x*20, y*20) for x,y in tile.qubits],
            fill = color_map[tile.color]
        ))
        drawing.add(drawing.circle(center=(tile.ancilla[0]*20, tile.ancilla[1]*20), r=3, fill='grey'))
        for x,y in tile.qubits:
            drawing.add(drawing.circle(center=(x*20, y*20), r=3, fill='black'))
    
    svg_data = drawing.tostring()
    display(SVG(svg_data))

draw_tiles(layout)

In [ ]:

circuit = stim.Circuit.generated(
      "color_code:memory_xyz",
      rounds=2,
      distance=3,
      after_clifford_depolarization=1e-3
)
print(str(circuit))

In [ ]:
circuit = stim.Circuit.generated(
      "color_code:memory_xyz",
      rounds=2,
      distance=5,
      after_clifford_depolarization=1e-3
)
print(str(circuit))